# Getting 1% seeds with HCP distance matrix 
(100 connectomes)

In [ ]:
import numpy as np 
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing.threshold_to_density import threshold_to_density

import numpy as np
from typing import Tuple, Callable
# import warnings
# warnings.filterwarnings('ignore')

# from gnm import generate_gnm_network # -> I can not get it easily to run, as I do not have the "imp" module... so I'll just bootstrap it. 


# Binary Generative Network Model (GNM) Grid Generator 
# Based on Betzel et al. methodology for brain network generation

# This implementation follows the generative modeling approach from:
# - Betzel et al. (2016) "Generative models of the human connectome"
# - Akarca et al. (2021) "A generative network model of neurodevelopmental diversity"


In [ ]:
connectomes_path = Path("/Users/adrian/Documents/01_projects/14_4D_lab/14_4D_lab_code/data/preprocessed/hcp_schaefer_100_dataset/01_connectomes/00_connectomes_density10.npy")
connectomes = np.load(connectomes_path)  # shape (100, 100, 100) (subjects, regions, regions)

preprocessed_base_path = Path("/Users/adrian/Documents/01_projects/14_4D_lab/14_4D_lab_code/data/preprocessed/hcp_schaefer_100_dataset/05_seeds_of_humans")
preprocessed_base_path.mkdir(parents=True, exist_ok=True)

debugger = None # 10 # or None # THIS IS IMPORTANT AS 100x100x100 is risky in terms of swapping dimensions... 
density = 2 # percent

In [ ]:
D = np.load("/Users/adrian/Documents/01_projects/14_4D_lab/14_4D_lab_code/data/preprocessed/hcp_schaefer_100_dataset/02_distance_matrices/distance_matrix_100.npy")

assert D.ndim == 2, "Distance matrix must be 2D"
assert D.shape[0] == D.shape[1], "Distance matrix must be square"
assert np.allclose(D, D.T), "Distance matrix must be symmetric"
assert np.allclose(np.diag(D), 0), "Diagonal must be zero"

In [ ]:
def create_parameter_grid(
    eta_range: Tuple[float, float],
    gamma_range: Tuple[float, float],
    n_points: int = 11
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Create a grid of eta and gamma parameters.
    
    Parameters:
    -----------
    eta_range : tuple
        (min, max) for eta parameter
    gamma_range : tuple
        (min, max) for gamma parameter
    n_points : int
        Number of points along each dimension
    
    Returns:
    --------
    eta_grid, gamma_grid : tuple of np.ndarray
        2D arrays of shape (n_points, n_points) with parameter values
    """
    eta_vals = np.linspace(eta_range[0], eta_range[1], n_points)
    gamma_vals = np.linspace(gamma_range[0], gamma_range[1], n_points)
    eta_grid, gamma_grid = np.meshgrid(eta_vals, gamma_vals)
    return eta_grid, gamma_grid


# ============================================================================
# Binary GNM Core Functions
# ============================================================================

def compute_edge_probability(
    D: np.ndarray,
    eta: float,
    gamma: float,
    rule: str = 'spatial'
) -> np.ndarray:
    """
    Compute edge formation probability based on distance and parameters.
    
    Parameters:
    -----------
    D : np.ndarray
        Distance matrix (n_nodes, n_nodes)
    eta : float
        Distance penalty parameter
    gamma : float
        Topology preference parameter
    rule : str
        Generative rule ('spatial', 'matching', etc.)
    
    Returns:
    --------
    P : np.ndarray
        Probability matrix (n_nodes, n_nodes)
    """
    n = D.shape[0]
    
    if rule == 'spatial':
        # P(i,j) ∝ exp(eta * D_ij)
        P = np.exp(eta * D)
    elif rule == 'matching':
        # Homophily: P(i,j) ∝ exp(eta * D_ij + gamma * |k_i - k_j|)
        # For initial network, assume uniform degree
        k = np.ones(n)
        K_diff = np.abs(k[:, None] - k[None, :])
        P = np.exp(eta * D + gamma * K_diff)
    else:
        raise ValueError(f"Unknown rule: {rule}")
    
    # Set diagonal to zero (no self-loops)
    np.fill_diagonal(P, 0)
    
    # Normalize
    P = P / (P.sum() + 1e-10)
    
    return P




In [ ]:
n_edges = int((density / 100) * (D.shape[0] * (D.shape[0] - 1)) / 2)
n_edges

In [ ]:
# eta_range = (-8.0, 3.0)
# gamma_range = (0.0, 1.0)
# (grid_eta, grid_gamma) = create_parameter_grid(eta_range, gamma_range, 100)


eta_range = (-8.0, 3.0)
gamma_range = (0.0, 1.0)
(grid_eta, grid_gamma) = create_parameter_grid(eta_range, gamma_range, 4)
# grid_eta

In [ ]:

def generate_binary_network(
    D: np.ndarray,
    eta: float,
    gamma: float,
    n_edges: int = None,
    rule: str = 'spatial',
    seed: int = None
) -> np.ndarray:
    """
    Generate a single binary network using GNM.
    
    Parameters:
    -----------
    D : np.ndarray
        Distance matrix
    eta : float
        Distance penalty parameter
    gamma : float
        Topology parameter
    n_edges : int, optional
        Target number of edges. If None, uses n*(n-1)/4
    rule : str
        Generative rule to use
    seed : int, optional
        Random seed for reproducibility
    
    Returns:
    --------
    A : np.ndarray
        Binary adjacency matrix (n_nodes, n_nodes)
    """
    if seed is not None:
        np.random.seed(seed)
    
    n = D.shape[0]
    
    if n_edges is None:
        # Default: ~25% density
        n_edges = n * (n - 1) // 4
    
    # Initialize empty network
    A = np.zeros((n, n), dtype=int)
    
    # Add edges sequentially
    for _ in range(n_edges):
        # Compute probabilities based on current state
        P = compute_edge_probability(D, eta, gamma, rule)
        
        # Mask already connected edges
        P[A > 0] = 0
        
        # Normalize
        P = P / P.sum()  # + 1e-10)
        
        # Sample edge
        idx = np.random.choice(n * n, p=P.ravel())
        i, j = divmod(idx, n)
        
        # Add edge (symmetric)
        A[i, j] = 1
        A[j, i] = 1
    
    return A

In [ ]:
conn = generate_binary_network(D, eta=-8.0, gamma=0.0, n_edges=n_edges, rule='spatial', seed=42)
conn

In [ ]:
number_conns_per_combination = 10

conn_arr = [] 
param_arr = [] # eta, gamma, connectome_number

for i in range(grid_eta.shape[0]):
    for j in range(grid_eta.shape[1]):
        for k in range(number_conns_per_combination):
            eta = grid_eta[i, j]
            gamma = grid_gamma[i, j]
            conn = generate_binary_network(D, eta=eta, gamma=gamma, n_edges=n_edges, rule='spatial', seed=42)
            conn_arr.append(conn)
            param_arr.append((eta, gamma, k))

conn_arr = np.array(conn_arr)
param_arr = np.array(param_arr)


In [ ]:
conn_arr 

In [ ]:
# save seed results 

np.save(preprocessed_base_path / f"seeds_density{density}_16_regions_conns.npy", conn_arr)
np.save(preprocessed_base_path / f"seeds_density{density}_16_regions_params.npy", param_arr)


In [ ]:
plt.imshow(conn, cmap='Greys')

# I think I dont need this anymore 

In [ ]:
eta = -8.0
gamma = 0.0
n_edges = None
rule = 'spatial'
seed = 42

if seed is not None:
    np.random.seed(seed)

n = D.shape[0]

if n_edges is None:
    # Default: ~25% density
    n_edges = n * (n - 1) // 4

# Initialize empty network
A = np.zeros((n, n), dtype=int)

# Add edges sequentially
for _ in range(n_edges):
    # Compute probabilities based on current state
    P = compute_edge_probability(D, eta, gamma, rule)
    
    # Mask already connected edges
    P[A > 0] = 0
    
    # Normalize
    P = P / (P.sum() + 1e-10)
    
    # Sample edge
    idx = np.random.choice(n * n, p=P.ravel())
    i, j = divmod(idx, n)
    
    # Add edge (symmetric)
    A[i, j] = 1
    A[j, i] = 1

In [ ]:

def generate_binary_network(
    D: np.ndarray,
    eta: float,
    gamma: float,
    n_edges: int = None,
    rule: str = 'spatial',
    seed: int = None
) -> np.ndarray:
    """
    Generate a single binary network using GNM.
    
    Parameters:
    -----------
    D : np.ndarray
        Distance matrix
    eta : float
        Distance penalty parameter
    gamma : float
        Topology parameter
    n_edges : int, optional
        Target number of edges. If None, uses n*(n-1)/4
    rule : str
        Generative rule to use
    seed : int, optional
        Random seed for reproducibility
    
    Returns:
    --------
    A : np.ndarray
        Binary adjacency matrix (n_nodes, n_nodes)
    """
    if seed is not None:
        np.random.seed(seed)
    
    n = D.shape[0]
    
    if n_edges is None:
        # Default: ~25% density
        n_edges = n * (n - 1) // 4
    
    # Initialize empty network
    A = np.zeros((n, n), dtype=int)
    
    # Add edges sequentially
    for _ in range(n_edges):
        # Compute probabilities based on current state
        P = compute_edge_probability(D, eta, gamma, rule)
        
        # Mask already connected edges
        P[A > 0] = 0
        
        # Normalize
        P = P / P.sum()  # + 1e-10)
        
        # Sample edge
        idx = np.random.choice(n * n, p=P.ravel())
        i, j = divmod(idx, n)
        
        # Add edge (symmetric)
        A[i, j] = 1
        A[j, i] = 1
    
    return A

In [ ]:
# # Compute probabilities based on current state
# P = compute_edge_probability(D, eta, gamma, rule)

# # Mask already connected edges
# P[A > 0] = 0

# # Normalize
# P = P / (P.sum()) #  + 1e-10)

# P.sum()

In [ ]:
# compute_edge_probability(D, eta, gamma, rule).sum()

In [ ]:
conn = generate_binary_network(D, eta=-8.0, gamma=0.0, n_edges=n_edges, rule='spatial', seed=42)
conn

In [ ]:
plt.imshow(conn, cmap='Greys')

In [ ]:
# Create parameter grid

eta_range = (-8, 3)
gamma_range = (0.1, 1.0)
n_points = 11
n_edges = None
rule = 'spatial'
seed = 42


eta_grid, gamma_grid = create_parameter_grid(eta_range, gamma_range, n_points)

n = D.shape[0]
n_networks = n_points * n_points
networks = np.zeros((n_networks, n, n), dtype=int)

print(f"Generating {n_networks} networks in {n_points}x{n_points} grid...")
print(f"Eta range: {eta_range}, Gamma range: {gamma_range}")
print(f"Network size: {n}x{n} nodes")

# Generate networks
idx = 0
for i in range(n_points):
    for j in range(n_points):
        eta = eta_grid[i, j]
        gamma = gamma_grid[i, j]
        
        networks[idx] = generate_binary_network(
            D, eta, gamma, n_edges, rule, seed=seed + idx
        )
        
        if (idx + 1) % 10 == 0:
            print(f"  Generated {idx + 1}/{n_networks} networks...")
        
        idx += 1

print("✓ Generation complete!")
# return networks, eta_grid, gamma_grid



In [ ]:

# ============================================================================
# Batch Generation Functions
# ============================================================================

def generate_gnm_grid(
    D: np.ndarray,
    eta_range: Tuple[float, float] = (-8, 3),
    gamma_range: Tuple[float, float] = (0.1, 1.0),
    n_points: int = 11,
    n_edges: int = None,
    rule: str = 'spatial',
    seed: int = 42
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Generate a grid of binary GNMs with varying parameters.
    
    Parameters:
    -----------
    D : np.ndarray
        Distance matrix
    eta_range : tuple
        Range for eta parameter
    gamma_range : tuple
        Range for gamma parameter
    n_points : int
        Grid resolution (11 for 11x11 grid)
    n_edges : int, optional
        Target edges per network
    rule : str
        Generative rule
    seed : int
        Random seed
    
    Returns:
    --------
    networks : np.ndarray
        Array of shape (n_points^2, n_nodes, n_nodes)
    eta_grid : np.ndarray
        2D grid of eta values
    gamma_grid : np.ndarray
        2D grid of gamma values
    """
    # Create parameter grid
    eta_grid, gamma_grid = create_parameter_grid(eta_range, gamma_range, n_points)
    
    n = D.shape[0]
    n_networks = n_points * n_points
    networks = np.zeros((n_networks, n, n), dtype=int)
    
    print(f"Generating {n_networks} networks in {n_points}x{n_points} grid...")
    print(f"Eta range: {eta_range}, Gamma range: {gamma_range}")
    print(f"Network size: {n}x{n} nodes")
    
    # Generate networks
    idx = 0
    for i in range(n_points):
        for j in range(n_points):
            eta = eta_grid[i, j]
            gamma = gamma_grid[i, j]
            
            networks[idx] = generate_binary_network(
                D, eta, gamma, n_edges, rule, seed=seed + idx
            )
            
            if (idx + 1) % 10 == 0:
                print(f"  Generated {idx + 1}/{n_networks} networks...")
            
            idx += 1
    
    print("✓ Generation complete!")
    return networks, eta_grid, gamma_grid


# ============================================================================
# Save/Load Functions
# ============================================================================

def save_gnm_results(
    networks: np.ndarray,
    eta_grid: np.ndarray,
    gamma_grid: np.ndarray,
    filepath: str = 'gnm_grid_results.npz'
) -> None:
    """Save generated networks and parameters."""
    np.savez_compressed(
        filepath,
        networks=networks,
        eta_grid=eta_grid,
        gamma_grid=gamma_grid
    )
    print(f"✓ Saved results to {filepath}")


def load_gnm_results(filepath: str = 'gnm_grid_results.npz') -> dict:
    """Load saved networks and parameters."""
    data = np.load(filepath)
    return {
        'networks': data['networks'],
        'eta_grid': data['eta_grid'],
        'gamma_grid': data['gamma_grid']
    }


# ============================================================================
# Utility Functions
# ============================================================================

def compute_network_statistics(A: np.ndarray) -> dict:
    """Compute basic statistics for a binary network."""
    n = A.shape[0]
    n_edges = A.sum() // 2  # Undirected
    density = n_edges / (n * (n - 1) / 2)
    degree = A.sum(axis=1)
    
    return {
        'n_nodes': n,
        'n_edges': n_edges,
        'density': density,
        'mean_degree': degree.mean(),
        'std_degree': degree.std()
    }


def print_grid_summary(networks: np.ndarray, eta_grid: np.ndarray, gamma_grid: np.ndarray) -> None:
    """Print summary statistics for the generated grid."""
    print(f"\n{'='*60}")
    print(f"GNM Grid Summary")
    print(f"{'='*60}")
    print(f"Grid size: {eta_grid.shape[0]}x{eta_grid.shape[1]}")
    print(f"Total networks: {networks.shape[0]}")
    print(f"Network dimensions: {networks.shape[1]}x{networks.shape[2]}")
    print(f"Eta range: [{eta_grid.min():.2f}, {eta_grid.max():.2f}]")
    print(f"Gamma range: [{gamma_grid.min():.2f}, {gamma_grid.max():.2f}]")
    
    # Sample network stats
    sample_stats = compute_network_statistics(networks[0])
    print(f"\nSample network statistics (first network):")
    for key, val in sample_stats.items():
        print(f"  {key}: {val:.4f}" if isinstance(val, float) else f"  {key}: {val}")
    print(f"{'='*60}\n")



In [ ]:
# 2. Generate GNM grid
networks, eta_grid, gamma_grid = generate_gnm_grid(
    D=D,
    eta_range=(-8, 3),
    gamma_range=(0.1, 1.0),
    n_points=11,
    n_edges=None,  # Auto: ~25% density
    rule='spatial',
    seed=42
)

# 3. Print summary
print_grid_summary(networks, eta_grid, gamma_grid)

# 4. Save results
save_gnm_results(networks, eta_grid, gamma_grid, 'gnm_grid_results.npz')

# 5. Verify shape
print(f"Final output shape: {networks.shape}")
print(f"Expected shape: (121, 100, 100)")

# Optional: Load and verify
# loaded = load_gnm_results('gnm_grid_results.npz')
# print(f"Loaded networks shape: {loaded['networks'].shape}")

# Old stuff: GNMs

In [ ]:

# ============================================================================
# Binary GNM Core Functions
# ============================================================================

def compute_edge_probability(
    D: np.ndarray,
    eta: float,
    gamma: float,
    rule: str = 'spatial'
) -> np.ndarray:
    """
    Compute edge formation probability based on distance and parameters.
    
    Parameters:
    -----------
    D : np.ndarray
        Distance matrix (n_nodes, n_nodes)
    eta : float
        Distance penalty parameter
    gamma : float
        Topology preference parameter
    rule : str
        Generative rule ('spatial', 'matching', etc.)
    
    Returns:
    --------
    P : np.ndarray
        Probability matrix (n_nodes, n_nodes)
    """
    n = D.shape[0]
    
    if rule == 'spatial':
        # P(i,j) ∝ exp(eta * D_ij)
        P = np.exp(eta * D)
    elif rule == 'matching':
        # Homophily: P(i,j) ∝ exp(eta * D_ij + gamma * |k_i - k_j|)
        # For initial network, assume uniform degree
        k = np.ones(n)
        K_diff = np.abs(k[:, None] - k[None, :])
        P = np.exp(eta * D + gamma * K_diff)
    else:
        raise ValueError(f"Unknown rule: {rule}")
    
    # Set diagonal to zero (no self-loops)
    np.fill_diagonal(P, 0)
    
    # Normalize
    P = P / (P.sum() + 1e-10)
    
    return P


def generate_binary_network(
    D: np.ndarray,
    eta: float,
    gamma: float,
    n_edges: int = None,
    rule: str = 'spatial',
    seed: int = None
) -> np.ndarray:
    """
    Generate a single binary network using GNM.
    
    Parameters:
    -----------
    D : np.ndarray
        Distance matrix
    eta : float
        Distance penalty parameter
    gamma : float
        Topology parameter
    n_edges : int, optional
        Target number of edges. If None, uses n*(n-1)/4
    rule : str
        Generative rule to use
    seed : int, optional
        Random seed for reproducibility
    
    Returns:
    --------
    A : np.ndarray
        Binary adjacency matrix (n_nodes, n_nodes)
    """
    if seed is not None:
        np.random.seed(seed)
    
    n = D.shape[0]
    
    if n_edges is None:
        # Default: ~25% density
        n_edges = n * (n - 1) // 4
    
    # Initialize empty network
    A = np.zeros((n, n), dtype=int)
    
    # Add edges sequentially
    for _ in range(n_edges):
        # Compute probabilities based on current state
        P = compute_edge_probability(D, eta, gamma, rule)
        
        # Mask already connected edges
        P[A > 0] = 0
        
        # Normalize
        P = P / (P.sum() + 1e-10)
        
        # Sample edge
        idx = np.random.choice(n * n, p=P.ravel())
        i, j = divmod(idx, n)
        
        # Add edge (symmetric)
        A[i, j] = 1
        A[j, i] = 1
    
    return A


# ============================================================================
# Batch Generation Functions
# ============================================================================

def generate_gnm_grid(
    D: np.ndarray,
    eta_range: Tuple[float, float] = (-8, 3),
    gamma_range: Tuple[float, float] = (0.1, 1.0),
    n_points: int = 11,
    n_edges: int = None,
    rule: str = 'spatial',
    seed: int = 42
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Generate a grid of binary GNMs with varying parameters.
    
    Parameters:
    -----------
    D : np.ndarray
        Distance matrix
    eta_range : tuple
        Range for eta parameter
    gamma_range : tuple
        Range for gamma parameter
    n_points : int
        Grid resolution (11 for 11x11 grid)
    n_edges : int, optional
        Target edges per network
    rule : str
        Generative rule
    seed : int
        Random seed
    
    Returns:
    --------
    networks : np.ndarray
        Array of shape (n_points^2, n_nodes, n_nodes)
    eta_grid : np.ndarray
        2D grid of eta values
    gamma_grid : np.ndarray
        2D grid of gamma values
    """
    # Create parameter grid
    eta_grid, gamma_grid = create_parameter_grid(eta_range, gamma_range, n_points)
    
    n = D.shape[0]
    n_networks = n_points * n_points
    networks = np.zeros((n_networks, n, n), dtype=int)
    
    print(f"Generating {n_networks} networks in {n_points}x{n_points} grid...")
    print(f"Eta range: {eta_range}, Gamma range: {gamma_range}")
    print(f"Network size: {n}x{n} nodes")
    
    # Generate networks
    idx = 0
    for i in range(n_points):
        for j in range(n_points):
            eta = eta_grid[i, j]
            gamma = gamma_grid[i, j]
            
            networks[idx] = generate_binary_network(
                D, eta, gamma, n_edges, rule, seed=seed + idx
            )
            
            if (idx + 1) % 10 == 0:
                print(f"  Generated {idx + 1}/{n_networks} networks...")
            
            idx += 1
    
    print("✓ Generation complete!")
    return networks, eta_grid, gamma_grid


# ============================================================================
# Save/Load Functions
# ============================================================================

def save_gnm_results(
    networks: np.ndarray,
    eta_grid: np.ndarray,
    gamma_grid: np.ndarray,
    filepath: str = 'gnm_grid_results.npz'
) -> None:
    """Save generated networks and parameters."""
    np.savez_compressed(
        filepath,
        networks=networks,
        eta_grid=eta_grid,
        gamma_grid=gamma_grid
    )
    print(f"✓ Saved results to {filepath}")


def load_gnm_results(filepath: str = 'gnm_grid_results.npz') -> dict:
    """Load saved networks and parameters."""
    data = np.load(filepath)
    return {
        'networks': data['networks'],
        'eta_grid': data['eta_grid'],
        'gamma_grid': data['gamma_grid']
    }


# ============================================================================
# Utility Functions
# ============================================================================

def compute_network_statistics(A: np.ndarray) -> dict:
    """Compute basic statistics for a binary network."""
    n = A.shape[0]
    n_edges = A.sum() // 2  # Undirected
    density = n_edges / (n * (n - 1) / 2)
    degree = A.sum(axis=1)
    
    return {
        'n_nodes': n,
        'n_edges': n_edges,
        'density': density,
        'mean_degree': degree.mean(),
        'std_degree': degree.std()
    }


def print_grid_summary(networks: np.ndarray, eta_grid: np.ndarray, gamma_grid: np.ndarray) -> None:
    """Print summary statistics for the generated grid."""
    print(f"\n{'='*60}")
    print(f"GNM Grid Summary")
    print(f"{'='*60}")
    print(f"Grid size: {eta_grid.shape[0]}x{eta_grid.shape[1]}")
    print(f"Total networks: {networks.shape[0]}")
    print(f"Network dimensions: {networks.shape[1]}x{networks.shape[2]}")
    print(f"Eta range: [{eta_grid.min():.2f}, {eta_grid.max():.2f}]")
    print(f"Gamma range: [{gamma_grid.min():.2f}, {gamma_grid.max():.2f}]")
    
    # Sample network stats
    sample_stats = compute_network_statistics(networks[0])
    print(f"\nSample network statistics (first network):")
    for key, val in sample_stats.items():
        print(f"  {key}: {val:.4f}" if isinstance(val, float) else f"  {key}: {val}")
    print(f"{'='*60}\n")



In [ ]:


eta_vals = np.linspace(eta_range[0], eta_range[1], n_points)
gamma_vals = np.linspace(gamma_range[0], gamma_range[1], n_points)
eta_grid, gamma_grid = np.meshgrid(eta_vals, gamma_vals)
return eta_grid, gamma_grid



In [ ]:
eta_range=(-8, 3)
gamma_range=(0.1, 1.0)
n_points=11
n_edges=None 
rule='spatial'
seed=42



# 3. Print summary
print_grid_summary(networks, eta_grid, gamma_grid)

# 4. Save results
save_gnm_results(networks, eta_grid, gamma_grid, 'gnm_grid_results.npz')

# 5. Verify shape
print(f"Final output shape: {networks.shape}")
print(f"Expected shape: (121, 100, 100)")

# Optional: Load and verify
# loaded = load_gnm_results('gnm_grid_results.npz')
# print(f"Loaded networks shape: {loaded['networks'].shape}")

In [ ]:
one_percent_connectomes = [] 

for conn in connectomes:  
    thres_conn, _ = threshold_to_density(
        conn_wei=conn,  # connection_matrix,
        # n_nodes=resolution,
        output_folder=None, # needs to be none - otherwise it overwrites (!) the preprocessed files. 
        density=density, # Minimum spanning tree with 99 edges exceeds desired connection density of 1% (49 edges). Cannot proceed with graph creation.
        conn_type="connectomes_resampled"
    )
    
    one_percent_connectomes.append(thres_conn)

In [ ]:
np.array(one_percent_connectomes).shape  # (100, 100, 100)
np.save(preprocessed_base_path / f"seeds_{density}_percent.npy", np.array(one_percent_connectomes))

### OLD: The following code uses thresholding of the empirical HCP networks - which is not sensible. 


In [ ]:
import numpy as np 
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing.threshold_to_density import threshold_to_density


In [ ]:
connectomes_path = Path("/Users/adrian/Documents/01_projects/14_4D_lab/14_4D_lab_code/data/preprocessed/hcp_schaefer_100_dataset/01_connectomes/00_connectomes_density10.npy")
connectomes = np.load(connectomes_path)  # shape (100, 100, 100) (subjects, regions, regions)

preprocessed_base_path = Path("/Users/adrian/Documents/01_projects/14_4D_lab/14_4D_lab_code/data/preprocessed/hcp_schaefer_100_dataset/05_seeds_of_humans")
preprocessed_base_path.mkdir(parents=True, exist_ok=True)

debugger = None # 10 # or None # THIS IS IMPORTANT AS 100x100x100 is risky in terms of swapping dimensions... 
density = 2 # percent

In [ ]:
one_percent_connectomes = [] 

for conn in connectomes:  
    thres_conn, _ = threshold_to_density(
        conn_wei=conn,  # connection_matrix,
        # n_nodes=resolution,
        output_folder=None, # needs to be none - otherwise it overwrites (!) the preprocessed files. 
        density=density, # Minimum spanning tree with 99 edges exceeds desired connection density of 1% (49 edges). Cannot proceed with graph creation.
        conn_type="connectomes_resampled"
    )
    
    one_percent_connectomes.append(thres_conn)

In [ ]:
np.array(one_percent_connectomes).shape  # (100, 100, 100)

In [ ]:
np.save(preprocessed_base_path / f"seeds_{density}_percent.npy", np.array(one_percent_connectomes))

# old stuff

In [ ]:
# Get distance matrix 

from scipy.spatial.distance import cdist

distance_matrix = cdist(coordinates, coordinates)
np.fill_diagonal(distance_matrix, 0) # np.inf)

distance_matrix_path = preprocessed_base_path / "02_distance_matrices"
distance_matrix_path.mkdir(parents=True, exist_ok=True)

np.save(distance_matrix_path / "distance_matrix_100.npy", distance_matrix)

In [ ]:

from __future__ import annotations

import argparse
import sys
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

import bct
from netneurotools.networks import threshold_network, struct_consensus
from scipy.spatial.distance import cdist



import networkx as nx
import numpy as np
from scipy.spatial.distance import pdist, squareform

import networkx as nx
import numpy as np

# Local imports from your codebase
# from notebook_setup import setup
from preprocessing.get_distance_matrix import get_distance_matrix_from_coords, get_distance_matrix_from_fiber_lengths
from analysis.structural_measures import analyze_connectomes

from src.preprocessing.preprocessing_setup import setup
from preprocessing.threshold_to_density import threshold_to_density


from src.preprocessing.utils import setup_paths, ensure_dir, save_numpy, save_dataframe # TODO: Remove this again, not needed. 
# ----------------------------
# Config & argument parsing
# ----------------------------
# @dataclass
# class PipelineConfig:
#     resolution: int = 68
#     goal_densities: Tuple[int, ...] = (10, 12, 14, 16, 18, 20)  # in percent
#     analyze_density: int = 10  # used for steps requiring a single binarized set
#     do_plots: bool = False # True

#     # Communication model and rich-club options for graph measures
#     comm_mode: str = "estrada_scaled"
#     rich_nodes_global: Optional[np.ndarray] = None
#     rich_top_percent: float = 0.20



# def parse_args() -> PipelineConfig:
#     p = argparse.ArgumentParser(description="Connectome preprocessing pipeline")
    
#     TARGET_TOTAL_NUMBER_OF_NODES = 50 # CHANGE THIS HERE
        
#     p.add_argument("--resolution", type=int, default=TARGET_TOTAL_NUMBER_OF_NODES, # CHANGE here
#                    help="Parcellation resolution for distance matrix & inputs (default: 68)")
#     p.add_argument("--goal-densities", type=int, nargs="+",
#                    default=[10, 12, 14, 16, 18, 20],
#                    help="One or more goal densities in PERCENT to retain during thresholding")
#     p.add_argument("--analyze-density", type=int, default=10,
#                    help="Which density (in percent) to use for analyses that require a single binarized set")
#     p.add_argument("--do_plots", action="store_true",
#                    help="Activate plotting (only save data)")
#     p.add_argument("--comm-mode", type=str, default="estrada_scaled",
#                    help="Communication model used in analyze_connectomes (default: estrada_scaled)")
#     p.add_argument("--rich-top-percent", type=float, default=0.20,
#                    help="Top fraction for rich club node selection (default: 0.20)")

#     args = p.parse_args()

#     return PipelineConfig(
#         resolution=args.resolution,
#         goal_densities=tuple(args.goal_densities),
#         analyze_density=args.analyze_density,
#         do_plots=args.do_plots,
#         comm_mode=args.comm_mode,
#         rich_top_percent=args.rich_top_percent,
#     )

# # def main(resolution = None):
# # cfg = parse_args()

goal_density = 10 
resolution = 100 
# cfg = PipelineConfig(
#         resolution=50, # args.resolution, # CHANGE THIS HERE ?
#         goal_densities=[10], # tuple(args.goal_densities),
#         analyze_density=10, # args.analyze_density,
#         do_plots=True, # args.do_plots,
#         comm_mode="estrada_scaled", # args.comm_mode,
#         rich_top_percent=0.2) 


# paths = setup_paths(dataset_name="suarez_MaMI_dataset") 



# # Setup paths
# base_path = paths["path_raw_data"] / "connectivity/mami"
# conn_dir = base_path / "conn_100"
# coords_dir = base_path / "coords"

# resolution = 50 # final resolution (if this is 100 <-> 50 nodes PER SIDE)

# # Intermediate paths 
# output_conn_dir = paths["path_00_preprocessed"] / f"connectomes_downsampled_to_{resolution}" # per SIDE -> so a total of 100. 
# output_coords_dir = paths["path_00_preprocessed"] / f"coords_downsampled_to_{resolution}"
# output_results_dir = paths["path_output_for_logs_and_plots"] / "results"
# output_distance_matr_resampled_dir = paths["path_00_preprocessed"] / f"dis_matrices_downsampled_to_{resolution}" # paths["path_02_distance_matrices"] 
# output_orig_dist_dir = paths["path_00_preprocessed"] / f"dis_matrices_orig"
# output_orig_conn_dir = paths["path_00_preprocessed"] / f"connectomes_orig"

# Create output directories
# output_conn_dir.mkdir(exist_ok=True)
# output_coords_dir.mkdir(exist_ok=True)
# output_results_dir.mkdir(exist_ok=True)
# output_distance_matr_resampled_dir.mkdir(exist_ok=True)
# output_orig_dist_dir.mkdir(exist_ok=True)
# output_orig_conn_dir.mkdir(exist_ok=True)


# Storage for binarized connectomes at each density
# all_bin_by_density = {density: [] for density in cfg.goal_densities}

connectomes_binarized = []

# Process each animal
for connection_matrix in connectomes:
    # Create binarized versions at different densities
    print(f"  Creating binarized versions:")
    # for density in cfg.goal_densities:

    
    # Use your existing function
    thres_conn, final_density = threshold_to_density(
        conn_wei=connection_matrix, 
        # n_nodes=resolution, 
        density=goal_density, # density, 
        output_folder=None, # temp_output, 
        conn_type="connectomes"
    )
    connectomes_binarized.append(thres_conn)

if debugger is not None:
    connectomes_binarized = connectomes_binarized[:debugger]
    
connectomes_binarized = np.array(connectomes_binarized)

# Analyze weighted connectome
results = analyze_connectomes(
    connectomes_binarized, 
    distance_matrix,
    comm_mode="estrada_scaled"
)

connectomes_save_path = preprocessed_base_path / "01_connectomes"
connectomes_save_path.mkdir(parents=True, exist_ok=True)
np.save(connectomes_save_path / f"00_connectomes_density{goal_density}.npy", connectomes_binarized)

# np.save(paths["path_02_distance_matrices"] / f"distance_matrix_{resolution}.npy", all_dist_downsampled)

# np.save(paths["path_01_connectomes"] / f"00_connectomes_orig.npy", all_orig_conn)
# np.save(paths["path_02_distance_matrices"] / f"distance_matrix_orig.npy", all_orig_dist_matr)

# # Save combined binarized arrays for each density
# print("\nSaving combined binarized arrays...")
# for density in cfg.goal_densities:
#     all_bin = np.stack(all_bin_by_density[density])  # Shape: (num_animals, 100, 100)
    
#     save_path = paths["path_01_connectomes"] / f"01_consensus_bin_density_{density}_percent_{resolution}.npy"
#     np.save(save_path, all_bin)
#     print(f"  Density {density}%: {all_bin.shape}")
    
#     # Clean up temporary files
#     temp_output = paths["path_00_preprocessed"] / f"temp_bin_{density}"
#     if temp_output.exists():
#         for file in temp_output.glob("*.npy"):
#             file.unlink()
#         temp_output.rmdir()

# Optionally: Analyze binarized connectomes at the specified density
# if cfg.analyze_density in cfg.goal_densities:
#     print(f"\nAnalyzing binarized connectomes at {cfg.analyze_density}% density...")
bin_results = []
# all_bin_analyze = all_bin_by_density[cfg.analyze_density]

for idx, (bin_conn) in enumerate(connectomes_binarized):
    results = analyze_connectomes(
        np.array([bin_conn]),  # Shape: (1, 100, 100)
        distance_matrix,
        comm_mode="estrada_scaled"
    )
    for result in results:
        bin_results.append(result)

# Save binary analysis results
bin_results_df = pd.DataFrame(bin_results)

path_03_graph_measures = preprocessed_base_path / "03_graph_measures"
path_03_graph_measures.mkdir(parents=True, exist_ok=True)

bin_results_df.to_csv(
    path_03_graph_measures / f"df_graph_measures_bin_{resolution}_density_{goal_density}_percent.csv",
    index=False
)
print(f"Binary analysis results saved: {bin_results_df.shape}")
print(path_03_graph_measures / f"df_graph_measures_bin_{resolution}_density_{goal_density}_percent.csv")

In [ ]:
# plt.imshow(distance_matrix, cmap='viridis')
plt.imshow(connectomes_binarized[8], cmap='viridis')

In [ ]:
connectomes_binarized.shape

In [ ]:
np.load("/Users/adrian/Documents/01_projects/14_4D_lab/14_4D_lab_code/output/gnm/hcp_schaefer_100_dataset/00_default_animal_0/generated_networks/net_eta-8.0_gamma-0.100000001490117_ruleMatchingIndex_id000.npy").sum()